Viz script for the result csvs of the different experiments

Note: Oyamada is renamed to FT LLM for all the plots

In [27]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

# ---- Setup ----
OUT_DIR = Path.cwd() / "complete_results"
OUT_DIR.mkdir(exist_ok=True)
DATA_DIR = Path.cwd()

sns.set_theme(style="whitegrid", context="talk")
# PALETTE = {"DATL-TAIA": "#1f77b4", "FT-LLM": "#d62728"}
PALETTE = {"DATL-TAIA": "#1f77b4", "FT-LLM": "#d62728", "MT-RNN": "#2ca02c"}

def rename_models(df: pd.DataFrame) -> pd.DataFrame:
    """Map Oyamada / Oyamada* -> FT-LLM."""
    df = df.copy()
    df["Model"] = df["Model"].replace({"Oyamada": "FT-LLM", "Oyamada*": "FT-LLM"})
    return df


print(DATA_DIR)


D:\01 Main\02 MASTER\0001\Y2\Q4_Prep\Q1_Q2_Thesis\code\gemppm_code\data_functions\updated_exp_results


Experiment 1: Scaling of the backbone

In [28]:
# Code with all the diagrams in 2 lines.
df1 = rename_models(pd.read_csv(DATA_DIR / "diff_backbone.csv"))
df1 = df1[df1["Backbone"] != "GPT2"]
models_in_data = df1["Model"].unique()

backbone_order = ["TinyLLM", "Qwen2.5", "Llama3.2"]
df1["Backbone"] = pd.Categorical(df1["Backbone"], categories=backbone_order, ordered=True)

backbone_markers = {
    "TinyLLM": "o",
    "Qwen2.5": "X",
    "Llama3.2": "s",
}

datasets = df1["Dataset"].unique()
n = len(datasets)
ncols = n
nrows = 2

# Bold everything
plt.rcParams.update({
    "font.size": 15,
    "axes.titlesize": 16,
    "axes.labelsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 14,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "figure.titleweight": "bold",
})

fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 5), sharey=False)

# Row 0: Accuracy vs Runtime
for col, dataset in enumerate(datasets):
    ax = axes[0, col]
    sub = df1[df1["Dataset"] == dataset]
    for model, color in PALETTE.items():
        for backbone, marker in backbone_markers.items():
            point = sub[(sub["Model"] == model) & (sub["Backbone"] == backbone)]
            if point.empty:
                continue
            ax.errorbar(
                point["Runtime_Hours"], point["NA_Acc"],
                yerr=point["NA_CI"], xerr=point["Runtime_Std"],
                fmt=marker, color=color, markersize=10,
                markeredgecolor="white", markeredgewidth=1.2,
                ecolor=color, capsize=4, linewidth=1.2,
            )
    ax.set_title(dataset)
    ax.set_xlabel("Runtime (hours)")
    if col == 0:
        ax.set_ylabel("NA Accuracy (%)")
    ax.grid(True, alpha=0.3)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")

# Row 1: MAE vs Runtime
for col, dataset in enumerate(datasets):
    ax = axes[1, col]
    sub = df1[df1["Dataset"] == dataset]
    for model, color in PALETTE.items():
        for backbone, marker in backbone_markers.items():
            point = sub[(sub["Model"] == model) & (sub["Backbone"] == backbone)]
            if point.empty:
                continue
            ax.errorbar(
                point["Runtime_Hours"], point["RT_MAE"],
                yerr=point["MAE_CI"], xerr=point["Runtime_Std"],
                fmt=marker, color=color, markersize=10,
                markeredgecolor="white", markeredgewidth=1.2,
                ecolor=color, capsize=4, linewidth=1.2,
            )
    ax.set_xlabel("Runtime (hours)")
    if col == 0:
        ax.set_ylabel("RT MAE")
    ax.grid(True, alpha=0.3)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")

# Legend
model_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color,
           markersize=11, label=model)
    for model, color in PALETTE.items()
    if model in models_in_data
]
backbone_handles = [
    Line2D([0], [0], marker=marker, color="w", markerfacecolor="gray",
           markersize=11, label=backbone)
    for backbone, marker in backbone_markers.items()
]
legend_handles = model_handles + backbone_handles



fig.legend(handles, labels, loc="upper center",
           bbox_to_anchor=(0.5, -0.02), ncol=len(labels),
           fontsize=11, prop={"weight": "bold"})


fig.suptitle("Accuracy and remaining-time MAE versus runtime across backbones", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "02_combined_runtime_tradeoff.png", dpi=150, bbox_inches="tight")
plt.close()

Experiment 2: Data Percentage

In [29]:
df2 = rename_models(pd.read_csv(DATA_DIR / "data_percentage.csv"))
df2["Training_Percentage"] = df2["Training_Percentage"].astype(str).str.rstrip("%").astype(float)

datasets = df2["Dataset"].unique()
n = len(datasets)

acc_ylims = {
    "BPI2012":    (60, 85),
    "BPI2017":    (70, 90),
    "BPI2020_DD": (70, 90),
    "BPI2015_2":  (25, 55),
}

# Auto-tighten MAE y-limits per dataset (with small padding)
mae_ylims = {}
for dataset in datasets:
    sub = df2[df2["Dataset"] == dataset]
    lo = (sub["RT_MAE"] - sub["MAE_CI"]).min()
    hi = (sub["RT_MAE"] + sub["MAE_CI"]).max()
    pad = (hi - lo) * 0.1
    mae_ylims[dataset] = (lo - pad, hi + pad)

# Bold everything
plt.rcParams.update({
    "font.size": 15,
    "axes.titlesize": 16,
    "axes.labelsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 14,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "figure.titleweight": "bold",
})

# Combined figure: 2 rows × n cols (accuracy top, MAE bottom)
fig, axes = plt.subplots(nrows=2, ncols=n, figsize=(3.2 * n, 5), sharey=False)

# Row 0: Accuracy
for col, dataset in enumerate(datasets):
    ax = axes[0, col]
    sub = df2[df2["Dataset"] == dataset]
    for model, color in PALETTE.items():
        m = sub[sub["Model"] == model].sort_values("Training_Percentage")
        x = m["Training_Percentage"].values * (100 / 65)
        y = m["NA_Acc"].values
        ax.plot(x, y, color=color, marker="o", linewidth=2, label=model)
        ax.fill_between(
            x, y - m["NA_CI"].values, y + m["NA_CI"].values,
            color=color, alpha=0.15, linewidth=0,
        )
    ax.set_title(dataset)
    ax.set_xlabel("Training data used (%)")
    if col == 0:
        ax.set_ylabel("NA Accuracy (%)")
    ax.set_xlim(25, 100)
    ax.set_ylim(acc_ylims.get(dataset, (0, 100)))
    # Bold tick labels
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")

# Row 1: MAE
for col, dataset in enumerate(datasets):
    ax = axes[1, col]
    sub = df2[df2["Dataset"] == dataset]
    for model, color in PALETTE.items():
        m = sub[sub["Model"] == model].sort_values("Training_Percentage")
        x = m["Training_Percentage"].values * (100 / 65)
        y = m["RT_MAE"].values
        ax.plot(x, y, color=color, marker="o", linewidth=2, label=model)
        ax.fill_between(
            x, y - m["MAE_CI"].values, y + m["MAE_CI"].values,
            color=color, alpha=0.15, linewidth=0,
        )
    ax.set_xlabel("Training data (%)")
    if col == 0:
        ax.set_ylabel("RT MAE")
    ax.set_xlim(25, 100)
    ax.set_ylim(mae_ylims[dataset])
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")


handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center",
           bbox_to_anchor=(0.5, -0.02), ncol=len(labels),
           fontsize=11, prop={"weight": "bold"})


fig.suptitle("Performance versus Percentage of training data", y=1.02)
fig.tight_layout()
plt.savefig(OUT_DIR / "03_combined_learning_curves.png", dpi=150, bbox_inches="tight")
plt.close()

Experiment 3: Prefix buckets

In [35]:
# import numpy as np

df3 = rename_models(pd.read_csv(DATA_DIR / "prefix_bucket.csv"))
df3["Prefix_Percentage"] = df3["Prefix_Percentage"].astype(str).str.rstrip("%")

prefix_order = ["10-20", "20-40", "40-60", "60-80", "80-100"]
df3["Prefix_Percentage"] = pd.Categorical(
    df3["Prefix_Percentage"], categories=prefix_order, ordered=True
)

datasets = df3["Dataset"].unique()
n = len(datasets)

# Auto-tighten y-limits per dataset (with 10% padding)
acc_ylims = {}
for dataset in datasets:
    sub = df3[df3["Dataset"] == dataset]
    lo = (sub["NA_Acc"] - sub["NA_CI"]).min()
    hi = (sub["NA_Acc"] + sub["NA_CI"]).max()
    pad = (hi - lo) * 0.1
    acc_ylims[dataset] = (max(0, lo - pad), hi + pad)

mae_ylims = {}
for dataset in datasets:
    sub = df3[df3["Dataset"] == dataset]
    lo = (sub["RT_MAE"] - sub["MAE_CI"]).min()
    hi = (sub["RT_MAE"] + sub["MAE_CI"]).max()
    pad = (hi - lo) * 0.1
    mae_ylims[dataset] = (max(0, lo - pad), hi + pad)

# Bold everything
plt.rcParams.update({
    "font.size": 15,
    "axes.titlesize": 16,
    "axes.labelsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 14,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "figure.titleweight": "bold",
})

# Bar layout
models = list(PALETTE.keys())
n_models = len(models)
bar_width = 0.8 / n_models  # total group width = 0.8
x_base = np.arange(len(prefix_order))

fig, axes = plt.subplots(nrows=2, ncols=n, figsize=(3.3 * n, 5), sharey=False)

def draw_bars(ax, sub, metric_col, ci_col):
    for i, model in enumerate(models):
        m = sub[sub["Model"] == model].sort_values("Prefix_Percentage")
        # Reindex to ensure all prefix buckets line up (in case any are missing)
        m = m.set_index("Prefix_Percentage").reindex(prefix_order)
        x = x_base + (i - (n_models - 1) / 2) * bar_width
        ax.bar(
            x, m[metric_col].values, width=bar_width,
            color=PALETTE[model], label=model,
            yerr=m[ci_col].values, capsize=3,
            error_kw={"elinewidth": 1.0, "ecolor": "black"},
            edgecolor="white", linewidth=0.5,
        )

# Row 0: Accuracy
for col, dataset in enumerate(datasets):
    ax = axes[0, col]
    sub = df3[df3["Dataset"] == dataset]
    draw_bars(ax, sub, "NA_Acc", "NA_CI")
    ax.set_title(dataset)
    ax.set_xlabel("")
    if col == 0:
        ax.set_ylabel("NA Accuracy (%)")
    ax.set_xticks(x_base)
    ax.set_xticklabels(prefix_order, rotation=20, ha="right")
    ax.set_ylim(acc_ylims[dataset])
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")

# Row 1: MAE
for col, dataset in enumerate(datasets):
    ax = axes[1, col]
    sub = df3[df3["Dataset"] == dataset]
    draw_bars(ax, sub, "RT_MAE", "MAE_CI")
    ax.set_xlabel("Prefix length (% of trace)")
    if col == 0:
        ax.set_ylabel("RT MAE")
    ax.set_xticks(x_base)
    ax.set_xticklabels(prefix_order, rotation=20, ha="right")
    ax.set_ylim(mae_ylims[dataset])
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")

# Shared legend at the bottom
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center",
           bbox_to_anchor=(0.5, -0.02), ncol=len(labels),
           fontsize=11, prop={"weight": "bold"})



# fig.suptitle("Learning Curves: Accuracy & Remaining-Time MAE vs Training Data", y=1.06)
fig.suptitle("Performance by prefix length bucket", y=1.02)
fig.tight_layout()
plt.savefig(OUT_DIR / "05_combined_prefix.png", dpi=150, bbox_inches="tight")
plt.close()